# General EDA: Subiculum Literature Dataset

**Goal:** Explore dataset characteristics, distribution patterns, and metadata statistics before diving into text analysis.

## Sections

1. Dataset Overview
2. Temporal Analysis
3. Journal Analysis
4. Author Analysis
5. Citation Network Basics
6. Keywords and MeSH Terms
7. Publication Types
8. Data Quality Assessment
9. Correlation Analysis

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from utils import (
    query_db, 
    save_figure, 
    save_dataframe,
    save_shareable_figure,
    save_shareable_table
)

# Plotting style
sns.set_style('whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

NOTEBOOK_NAME = 'general_eda'

## 1. Dataset Overview

Basic statistics about the dataset:
- Total papers count
- Year range (min, max, span)
- Papers with/without abstracts
- Papers with/without DOI
- Papers with open access
- Average text length

In [ ]:
# Load basic dataset statistics
overview_sql = """
SELECT 
  COUNT(*) as total_papers,
  MIN(pub_year) as earliest_year,
  MAX(pub_year) as latest_year,
  COUNT(CASE WHEN abstract IS NOT NULL AND abstract != '' THEN 1 END) as papers_with_abstract,
  COUNT(CASE WHEN doi IS NOT NULL AND doi != '' THEN 1 END) as papers_with_doi,
  COUNT(CASE WHEN pmc_id IS NOT NULL AND pmc_id != '' THEN 1 END) as papers_with_pmc,
  AVG(LENGTH(COALESCE(title, '') || ' ' || COALESCE(abstract, ''))) as avg_text_length
FROM papers;
"""

overview = query_db(overview_sql)
overview

In [ ]:
total = overview['total_papers'].iloc[0]
with_abstract = overview['papers_with_abstract'].iloc[0]
with_doi = overview['papers_with_doi'].iloc[0]
with_pmc = overview['papers_with_pmc'].iloc[0]
year_span = overview['latest_year'].iloc[0] - overview['earliest_year'].iloc[0]

print(f"Dataset Overview")
print(f"="*50)
print(f"Total papers: {total:,}")
print(f"Year range: {overview['earliest_year'].iloc[0]} - {overview['latest_year'].iloc[0]} ({year_span} years)")
print(f"Papers with abstracts: {with_abstract:,} ({100*with_abstract/total:.1f}%)")
print(f"Papers with DOI: {with_doi:,} ({100*with_doi/total:.1f}%)")
print(f"Papers with PMC ID: {with_pmc:,} ({100*with_pmc/total:.1f}%)")
print(f"Average text length: {overview['avg_text_length'].iloc[0]:.0f} characters")

In [ ]:
save_dataframe(overview, 'dataset_overview.csv', NOTEBOOK_NAME)

## 2. Temporal Analysis

Analyze publication trends over time:
- Papers by year
- Papers by decade
- Publication acceleration
- Recent trends

In [ ]:
papers_by_year = query_db("""
SELECT pub_year, COUNT(*) as count
FROM papers
WHERE pub_year IS NOT NULL
GROUP BY pub_year
ORDER BY pub_year;
""")

papers_by_year.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(papers_by_year['pub_year'], papers_by_year['count'], linewidth=2, marker='o', markersize=3)
ax.set_xlabel('Publication Year')
ax.set_ylabel('Number of Papers')
ax.set_title('Subiculum Research Publications Over Time (1966-2025)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Annotate key findings
max_year = papers_by_year.loc[papers_by_year['count'].idxmax()]
ax.annotate(f"Peak: {max_year['count']:.0f} papers\nin {max_year['pub_year']:.0f}",
            xy=(max_year['pub_year'], max_year['count']),
            xytext=(max_year['pub_year']-10, max_year['count']+20),
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
            fontsize=10, color='red')

plt.tight_layout()
save_figure(fig, 'papers_by_year.png', NOTEBOOK_NAME)
plt.show()

In [ ]:
papers_by_decade = query_db("""
SELECT 
    (pub_year / 10) * 10 as decade,
    COUNT(*) as count
FROM papers
WHERE pub_year IS NOT NULL
GROUP BY decade
ORDER BY decade;
""")

papers_by_decade['decade_label'] = papers_by_decade['decade'].astype(int).astype(str) + 's'
papers_by_decade

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(papers_by_decade['decade_label'], papers_by_decade['count'], 
              color=sns.color_palette('husl', len(papers_by_decade)))
ax.set_xlabel('Decade')
ax.set_ylabel('Number of Papers')
ax.set_title('Subiculum Research by Decade', fontsize=14, fontweight='bold')

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height):,}',
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
save_figure(fig, 'papers_by_decade.png', NOTEBOOK_NAME)
plt.show()

In [ ]:
papers_by_year['rolling_avg'] = papers_by_year['count'].rolling(window=5, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(papers_by_year['pub_year'], papers_by_year['count'], 
        alpha=0.3, linewidth=1, label='Annual count', color='gray')
ax.plot(papers_by_year['pub_year'], papers_by_year['rolling_avg'], 
        linewidth=3, label='5-year moving average', color='#2E86AB')
ax.set_xlabel('Publication Year')
ax.set_ylabel('Number of Papers')
ax.set_title('Publication Acceleration: 5-Year Moving Average', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
save_figure(fig, 'publication_acceleration.png', NOTEBOOK_NAME)
plt.show()

In [ ]:
recent_years = papers_by_year[papers_by_year['pub_year'] >= papers_by_year['pub_year'].max() - 10]

print(f"Last 10 Years Trend")
print(f"="*50)
print(recent_years[['pub_year', 'count']].to_string(index=False))
print(f"\nTotal papers (last 10 years): {recent_years['count'].sum():,}")
print(f"Average per year: {recent_years['count'].mean():.1f}")

In [ ]:
save_dataframe(papers_by_year, 'papers_by_year.csv', NOTEBOOK_NAME)
save_dataframe(papers_by_decade, 'papers_by_decade.csv', NOTEBOOK_NAME)

## 3. Journal Analysis

Analyze journal distribution:
- Top journals by paper count
- Journal diversity
- Open access rates by journal

In [ ]:
completeness = query_db("""
  SELECT 
      COUNT(*) as total_papers,
      COUNT(title) as has_title,
      COUNT(abstract) as has_abstract,
      COUNT(doi) as has_doi,
      COUNT(pmc_id) as has_pmc,
      COUNT(pub_year) as has_year,
      COUNT(journal_name) as has_journal
  FROM papers;
  """)
completeness

In [ ]:
total = completeness['total_papers'].iloc[0]
completeness_pct = pd.DataFrame({
  'Field': ['Title', 'Abstract', 'DOI', 'PMC ID', 'Year', 'Journal'],
  'Count': [
      completeness['has_title'].iloc[0],
      completeness['has_abstract'].iloc[0],
      completeness['has_doi'].iloc[0],
      completeness['has_pmc'].iloc[0],
      completeness['has_year'].iloc[0],
      completeness['has_journal'].iloc[0]
  ]
})
completeness_pct['Percentage'] = 100 * completeness_pct['Count'] / total

completeness_pct

In [ ]:
 top_journals = query_db("""
SELECT 
  journal_name,
  COUNT(*) as paper_count,
  COUNT(CASE WHEN pmc_id IS NOT NULL AND pmc_id != '' THEN 1 END) as pmc_count,
  ROUND(100.0 * COUNT(CASE WHEN pmc_id IS NOT NULL AND pmc_id != '' THEN 1 END) / COUNT(*), 1) as pmc_percentage
FROM papers
WHERE journal_name IS NOT NULL AND journal_name != ''
GROUP BY journal_name
ORDER BY paper_count DESC
LIMIT 30;
""")

top_journals

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
y_pos = np.arange(len(top_journals))
ax.barh(y_pos, top_journals['paper_count'], color='steelblue')
ax.set_yticks(y_pos)
ax.set_yticklabels(top_journals['journal_name'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Number of Papers')
ax.set_title('Top 30 Journals Publishing Subiculum Research', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

for i, v in enumerate(top_journals['paper_count']):
    ax.text(v + 5, i, str(v), va='center', fontsize=8)

plt.tight_layout()
save_figure(fig, 'top_journals.png', NOTEBOOK_NAME)
plt.show()

In [ ]:
journal_diversity = query_db("""
SELECT 
    COUNT(DISTINCT journal_name) as unique_journals,
    COUNT(*) as total_papers
FROM papers
WHERE journal_name IS NOT NULL AND journal_name != '';
""")

print(f" Journal Diversity")
print(f"="*50)
print(f"Unique journals: {journal_diversity['unique_journals'].iloc[0]:,}")
print(f"Total papers: {journal_diversity['total_papers'].iloc[0]:,}")
print(f"Average papers per journal: {journal_diversity['total_papers'].iloc[0] / journal_diversity['unique_journals'].iloc[0]:.1f}")

# Concentration metric: % of papers in top 10 journals
top10_count = top_journals.head(10)['paper_count'].sum()
concentration = 100 * top10_count / total
print(f"\nTop 10 journals account for: {concentration:.1f}% of all papers")

In [ ]:
save_dataframe(top_journals, 'top_journals.csv', NOTEBOOK_NAME)

## 4. Author Analysis

Analyze author contributions:
- Most prolific authors
- Author contribution over time
- Single vs multi-author papers
- Average authors per paper over time

In [ ]:
# Most prolific authors (top 50)
top_authors = query_db("""
SELECT 
    a.last_name,
    a.fore_name,
    a.initials,
    COUNT(DISTINCT pa.pmid) as paper_count,
    MIN(p.pub_year) as first_publication,
    MAX(p.pub_year) as latest_publication,
    MAX(p.pub_year) - MIN(p.pub_year) as years_active
FROM authors a
JOIN paper_authors pa ON a.author_id = pa.author_id
JOIN papers p ON pa.pmid = p.pmid
GROUP BY a.author_id
ORDER BY paper_count DESC
LIMIT 50;
""")

# Create display name
top_authors['author_name'] = (top_authors['last_name'] + ', ' + 
                               top_authors['fore_name'].fillna(top_authors['initials']))
top_authors

In [ ]:
display_authors = top_authors.head(30).copy()
display_authors.index = range(1, len(display_authors) + 1)
display_authors.index.name = 'Rank'


author_table = display_authors[['author_name', 'paper_count', 'first_publication', 'latest_publication', 'years_active']].copy()
author_table.columns = ['Author', 'Papers', 'First Pub', 'Latest Pub', 'Years Active']

print("Top 30 Most Prolific Authors in Subiculum Research")
print("="*80)
print(author_table.to_string())
print("="*80)

In [ ]:
top_100_authors = top_authors.head(100).copy()
top_100_authors.index = range(1, len(top_100_authors) + 1)
top_100_authors.index.name = 'Rank'

author_csv = top_100_authors[['author_name', 'paper_count', 'first_publication', 'latest_publication', 'years_active']].copy()
author_csv.columns = ['Author', 'Papers', 'First_Publication', 'Latest_Publication', 'Years_Active']

save_dataframe(author_csv, 'top_100_authors.csv', NOTEBOOK_NAME)

In [ ]:
authors_per_paper = query_db("""
SELECT 
  author_count,
  COUNT(*) as paper_count
FROM (
  SELECT pmid, COUNT(*) as author_count
  FROM paper_authors
  GROUP BY pmid
)
GROUP BY author_count
ORDER BY author_count;
""")

# Summary statistics
print("Author Count Distribution")
print("="*50)
print(f"Single-author papers: {authors_per_paper[authors_per_paper['author_count']==1]['paper_count'].sum():,}")
print(f"Multi-author papers: {authors_per_paper[authors_per_paper['author_count']>1]['paper_count'].sum():,}")
print(f"Median authors per paper: {authors_per_paper['author_count'].median():.0f}")
print(f"Mean authors per paper: {(authors_per_paper['author_count'] * authors_per_paper['paper_count']).sum() / authors_per_paper['paper_count'].sum():.1f}")
print(f"Max authors on a single paper: {authors_per_paper['author_count'].max()}")

print("\nDistribution (first 20):")
print(authors_per_paper.head(20).to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left plot: Full distribution (bar chart)
ax1.bar(authors_per_paper['author_count'], authors_per_paper['paper_count'],
      color='coral', edgecolor='black', linewidth=0.5, alpha=0.8)
ax1.set_xlabel('Number of Authors per Paper')
ax1.set_ylabel('Number of Papers')
ax1.set_title('Full Distribution of Authors per Paper', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')
ax1.set_xlim(0, min(50, authors_per_paper['author_count'].max()))  # Cap at 50 for readability

# Right plot: Cumulative distribution
cumulative = authors_per_paper.copy()
cumulative['cumulative_papers'] = cumulative['paper_count'].cumsum()
cumulative['cumulative_pct'] = 100 * cumulative['cumulative_papers'] / cumulative['paper_count'].sum()

ax2.plot(cumulative['author_count'], cumulative['cumulative_pct'],
       linewidth=2.5, color='#2E86AB', marker='o', markersize=3)
ax2.axhline(y=50, color='red', linestyle='--', linewidth=1, alpha=0.5, label='50th percentile')
ax2.axhline(y=90, color='orange', linestyle='--', linewidth=1, alpha=0.5, label='90th percentile')
ax2.set_xlabel('Number of Authors per Paper')
ax2.set_ylabel('Cumulative Percentage of Papers (%)')
ax2.set_title('Cumulative Distribution', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend()
ax2.set_xlim(0, min(30, authors_per_paper['author_count'].max()))

plt.tight_layout()
save_figure(fig, 'author_count_distribution.png', NOTEBOOK_NAME)
plt.show()

In [ ]:
avg_authors_by_year = query_db("""
SELECT 
    p.pub_year,
    AVG(author_counts.count) as avg_authors
FROM papers p
JOIN (
    SELECT pmid, COUNT(*) as count
    FROM paper_authors
    GROUP BY pmid
) author_counts ON p.pmid = author_counts.pmid
WHERE p.pub_year IS NOT NULL
GROUP BY p.pub_year
ORDER BY p.pub_year;
""")

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(avg_authors_by_year['pub_year'], avg_authors_by_year['avg_authors'], 
        linewidth=2, marker='o', markersize=3, color='#A23B72')
ax.set_xlabel('Publication Year')
ax.set_ylabel('Average Number of Authors')
ax.set_title('Average Authors per Paper Over Time', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
save_figure(fig, 'avg_authors_over_time.png', NOTEBOOK_NAME)
plt.show()

In [ ]:
save_dataframe(top_authors, 'top_authors.csv', NOTEBOOK_NAME)
save_dataframe(authors_per_paper, 'author_count_distribution.csv', NOTEBOOK_NAME)


## 5. Citation Network Basics

Basic citation statistics:
- Papers with vs without citations
- Most cited papers
- Citation distribution

In [ ]:
citation_stats = query_db("""
SELECT 
  COUNT(DISTINCT citing_pmid) as papers_with_citations,
  COUNT(*) as total_citation_links,
  ROUND(AVG(citation_count), 1) as avg_citations_per_paper
FROM (
  SELECT citing_pmid, COUNT(*) as citation_count
  FROM citations
  GROUP BY citing_pmid
);
""")

total_papers = query_db("SELECT COUNT(*) as count FROM papers")['count'].iloc[0]
papers_with_refs = citation_stats['papers_with_citations'].iloc[0]
total_refs = citation_stats['total_citation_links'].iloc[0]

print(f" Citation Statistics")
print(f"="*50)
print(f"Papers with reference lists: {papers_with_refs:,} ({100*papers_with_refs/total_papers:.1f}%)")
print(f"Total reference citations: {total_refs:,}")
print(f"Average references per paper (for papers with refs): {citation_stats['avg_citations_per_paper'].iloc[0]:.1f}")
print(f" Note: Only {100*papers_with_refs/total_papers:.1f}% of papers have reference lists.")

In [ ]:
# Most cited papers (within dataset...)
most_cited = query_db("""
SELECT 
    p.pmid,
    p.title,
    p.pub_year,
    p.journal_name,
    COUNT(c.citing_pmid) as times_cited
FROM papers p
LEFT JOIN citations c ON p.pmid = c.cited_pmid
GROUP BY p.pmid
ORDER BY times_cited DESC
LIMIT 20;
""")

most_cited

In [ ]:
refs_by_decade = query_db("""
SELECT 
  (p.pub_year / 10) * 10 as decade,
  COUNT(DISTINCT p.pmid) as total_papers,
  COUNT(DISTINCT c.citing_pmid) as papers_with_refs,
  ROUND(100.0 * COUNT(DISTINCT c.citing_pmid) / COUNT(DISTINCT p.pmid), 1) as pct_with_refs
FROM papers p
LEFT JOIN citations c ON p.pmid = c.citing_pmid
WHERE p.pub_year IS NOT NULL
GROUP BY decade
ORDER BY decade;
""")

print("\nReference List Availability by Decade")
print("="*60)
print(refs_by_decade.to_string(index=False))


In [ ]:
# this seems off.... lets do some chekcing

print("Citation Data Quality Check")
print("="*60)

# Check total papers vs papers in citation table
citation_check = query_db("""
SELECT 
  (SELECT COUNT(*) FROM papers) as total_papers,
  (SELECT COUNT(DISTINCT citing_pmid) FROM citations) as papers_in_citation_table,
  (SELECT COUNT(DISTINCT cited_pmid) FROM citations) as unique_papers_cited,
  (SELECT COUNT(*) FROM citations) as total_citation_records
""")
print("\nCitation Table Overview:")
print(citation_check.to_string(index=False))

# Sample recent papers without citations
recent_no_cites = query_db("""
SELECT 
  p.pmid,
  p.title,
  p.pub_year,
  p.journal_name
FROM papers p
LEFT JOIN citations c ON p.pmid = c.citing_pmid
WHERE p.pub_year >= 2020 
AND c.citing_pmid IS NULL
ORDER BY p.pub_year DESC
LIMIT 10;
""")

print("\n\nSample 2020s Papers WITHOUT Citation Data:")
print("="*60)
for idx, row in recent_no_cites.iterrows():
  print(f"\nPMID: {row['pmid']} ({row['pub_year']})")
  print(f"Title: {row['title'][:80]}...")
  print(f"Journal: {row['journal_name']}")


# Check if citations were even attempted to be fetched
fetch_status = query_db("""
SELECT 
  fetch_status,
  COUNT(*) as paper_count
FROM papers
GROUP BY fetch_status;
""")

print("\n\n Paper Fetch Status:")
print("="*60)
print(fetch_status.to_string(index=False))

**There seems ot be an issue with the ref list for about 58% of our papers, we will explre additional datasets and APIS for the rest....**

In [ ]:
citation_distribution = query_db("""
SELECT 
    times_cited,
    COUNT(*) as paper_count
FROM (
    SELECT 
        p.pmid,
        COUNT(c.citing_pmid) as times_cited
    FROM papers p
    LEFT JOIN citations c ON p.pmid = c.cited_pmid
    GROUP BY p.pmid
)
GROUP BY times_cited
ORDER BY times_cited;
""")

# Plot citation distribution (log scale)
fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(citation_distribution['times_cited'], citation_distribution['paper_count'], 
       color='#F18F01', edgecolor='black', linewidth=0.5)
ax.set_xlabel('Number of Times Cited (within dataset)')
ax.set_ylabel('Number of Papers (log scale)')
ax.set_yscale('log')
ax.set_title('Citation Distribution', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
save_figure(fig, 'citation_distribution.png', NOTEBOOK_NAME)
plt.show()

In [ ]:
save_dataframe(most_cited, 'most_cited_papers.csv', NOTEBOOK_NAME)
save_dataframe(citation_distribution, 'citation_distribution.csv', NOTEBOOK_NAME)

**There was an issue with PubMed not having ref lists for most papers, so I enrched the data by adding scripts for API calls for crossref and semantic scholar, which had most of them.**

## 6. Keywords and MeSH Terms

Analyze keywords and MeSH terms:
- Most common MeSH terms
- Most common author keywords

**There was an issue where I forgot to add these to my intial DB pull... oops! so I wrote a one off script to add them, and fixed the bug in the ETL script.**

In [ ]:
# Most common MeSH terms
top_mesh = query_db("""
SELECT 
  mt.descriptor_name as descriptor,
  COUNT(DISTINCT pmt.pmid) as paper_count
FROM mesh_terms mt
JOIN paper_mesh_terms pmt ON mt.mesh_id = pmt.mesh_id
WHERE mt.descriptor_name IS NOT NULL
GROUP BY mt.descriptor_name
ORDER BY paper_count DESC
LIMIT 20;
""")

top_mesh

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
display_mesh = top_mesh.head(20)
y_pos = np.arange(len(display_mesh))
ax.barh(y_pos, display_mesh['paper_count'], color='#6A994E')
ax.set_yticks(y_pos)
ax.set_yticklabels(display_mesh['descriptor'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Number of Papers')
ax.set_title('Top 20 MeSH Terms in Subiculum Research', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

for i, v in enumerate(display_mesh['paper_count']):
    ax.text(v + 10, i, str(v), va='center', fontsize=8)

plt.tight_layout()
save_figure(fig, 'top_mesh_terms.png', NOTEBOOK_NAME)
plt.show()

In [ ]:
top_keywords = query_db("""
SELECT 
  k.keyword,
  COUNT(DISTINCT pk.pmid) as paper_count
FROM keywords k
JOIN paper_keywords pk ON k.keyword_id = pk.keyword_id
GROUP BY k.keyword_id, k.keyword
ORDER BY paper_count DESC
LIMIT 30;
""")

top_keywords.head(20)

In [ ]:
save_dataframe(top_mesh, 'top_mesh_terms.csv', NOTEBOOK_NAME)
save_dataframe(top_keywords, 'top_keywords.csv', NOTEBOOK_NAME)

## 7. Publication Types

Analyze publication types:
- Distribution of publication types
- Reviews vs original research over time

In [ ]:
pub_types = query_db("""
SELECT 
  pt.pub_type_name as publication_type,
  COUNT(DISTINCT ppt.pmid) as paper_count
FROM publication_types pt
JOIN paper_publication_types ppt ON pt.pub_type_id = ppt.pub_type_id
GROUP BY pt.pub_type_id, pt.pub_type_name
ORDER BY paper_count DESC
LIMIT 20;
""")

pub_types

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
y_pos = np.arange(len(pub_types))
ax.barh(y_pos, pub_types['paper_count'], color='#BC4749')
ax.set_yticks(y_pos)
ax.set_yticklabels(pub_types['publication_type'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Number of Papers')
ax.set_title('Publication Types Distribution', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
save_figure(fig, 'publication_types.png', NOTEBOOK_NAME)
plt.show()

In [ ]:
save_dataframe(pub_types, 'publication_types.csv', NOTEBOOK_NAME)

## 8. Data Quality Assessment

Assess data completeness:
- Missing data by field
- Completeness heatmap (Todo)

In [ ]:
# Data completeness
completeness = query_db("""
SELECT 
  COUNT(*) as total_papers,
  COUNT(title) as has_title,
  COUNT(abstract) as has_abstract,
  COUNT(doi) as has_doi,
  COUNT(pub_year) as has_year,
  COUNT(journal_name) as has_journal,
  COUNT(pmc_id) as has_pmc
FROM papers;
""")

# Calculate percentages
total = completeness['total_papers'].iloc[0]
completeness_pct = pd.DataFrame({
  'Field': ['Title', 'Abstract', 'DOI', 'Year', 'Journal', 'PMC ID'],
  'Count': [
      completeness['has_title'].iloc[0],
      completeness['has_abstract'].iloc[0],
      completeness['has_doi'].iloc[0],
      completeness['has_year'].iloc[0],
      completeness['has_journal'].iloc[0],
      completeness['has_pmc'].iloc[0]  # Changed from has_oa_flag
  ]
})
completeness_pct['Percentage'] = 100 * completeness_pct['Count'] / total

completeness_pct

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(completeness_pct['Field'], completeness_pct['Percentage'], 
              color='#386641', edgecolor='black', linewidth=1)
ax.set_ylabel('Completeness (%)')
ax.set_title('Data Completeness by Field', fontsize=14, fontweight='bold')
ax.set_ylim(0, 105)
ax.axhline(y=100, color='red', linestyle='--', linewidth=1, alpha=0.5)
ax.grid(True, alpha=0.3, axis='y')

# Add percentage labels
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 1,
            f'{height:.1f}%',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
save_figure(fig, 'data_completeness.png', NOTEBOOK_NAME)
plt.show()

In [ ]:
save_dataframe(completeness_pct, 'data_completeness.csv', NOTEBOOK_NAME)

## 9. Summary Statistics Table

Generate a comprehensive summary table for the report.

In [ ]:
with_pmc = completeness['has_pmc'].iloc[0]

summary = pd.DataFrame({
  'Metric': [
      'Total Papers',
      'Year Range',
      'Unique Journals',
      'Unique Authors',
      'Papers with Abstracts',
      'Papers with DOI',
      'Papers with PMC ID',
      'Total Citations',
      'Avg Authors per Paper',
      'Avg Citations per Paper'
  ],
  'Value': [
      f"{total:,}",
      f"{overview['earliest_year'].iloc[0]:.0f} - {overview['latest_year'].iloc[0]:.0f}",
      f"{journal_diversity['unique_journals'].iloc[0]:,}",
      f"{query_db('SELECT COUNT(DISTINCT author_id) FROM authors')['COUNT(DISTINCT author_id)'].iloc[0]:,}",
      f"{with_abstract:,} ({100*with_abstract/total:.1f}%)",
      f"{with_doi:,} ({100*with_doi/total:.1f}%)",
      f"{with_pmc:,} ({100*with_pmc/total:.1f}%)",
      f"{citation_stats['total_citation_links'].iloc[0]:,}",
      f"{avg_authors_by_year['avg_authors'].mean():.1f}",
      f"{citation_stats['avg_citations_per_paper'].iloc[0]:.1f}"
  ]
})

print("\nSUMMARY STATISTICS")
print("="*60)
print(summary.to_string(index=False))
print("="*60)

In [ ]:
save_dataframe(summary, 'summary_statistics.csv', NOTEBOOK_NAME)

print("\n EDA Complete!")
print(f"All outputs saved to: notebooks/eda/")